In [6]:
import pickle
from load_and_filter import load_pipeline

# data = load_pipeline(
#     pkl1         = "/home/aarghavan/nasShare/projects/labdul/Data_siegel/all_data_siegel_0.025.pkl",
#     pkl2         = "/home/aarghavan/nasShare/projects/labdul/Data_siegel/new_13_sessions/all_tasks/all_new_13_tasks.pkl",
#     behavior_csv = "/home/aarghavan/aslan/data/behavior_all.csv",
#     subject      = None,                            # None = both, "Paula", "Rex"
#     nMapStim     = None,                            # None = all;  or [6, 8]
#     max_iti      = None,                            # None = no ITI filter
#     original_bin = 0.025,
#     target_bin   = 0.1,
#     output_path  = "/home/aarghavan/aslan/data/test.pkl",  # None = don't save, "/home/aarghavan/aslan/data/test.pkl"
# )

data = load_pipeline(
    pkl1               = "/home/aarghavan/nasShare/projects/labdul/Data_siegel/all_data_siegel_0.025.pkl",
    pkl2               = "/home/aarghavan/nasShare/projects/labdul/Data_siegel/new_13_sessions/all_tasks/all_new_13_tasks.pkl",
    behavior_csv       = "/home/aarghavan/aslan/data/behavior_all.csv",
    subject            = None,
    nMapStim           = None,
    max_iti            = None,
    original_bin       = 0.025,
    use_sliding_window = True,
    window_s           = 0.1,
    step_s             = 0.025,
    output_path        = "/home/aarghavan/aslan/data/test.pkl",
)



[   0.0s]  Loading PKL files ...
Loading PKL1: /home/aarghavan/nasShare/projects/labdul/Data_siegel/all_data_siegel_0.025.pkl
Loading PKL2: /home/aarghavan/nasShare/projects/labdul/Data_siegel/new_13_sessions/all_tasks/all_new_13_tasks.pkl
  PKL1: 53 sessions
  PKL2: 13 sessions
  Combined: 66 sessions total
         → 66 total sessions loaded

[  46.4s]  Extracting delsac sessions ...


Extracting delsac sessions: 100%|██████████| 66/66 [00:06<00:00, 10.10it/s]


  66 delsac sessions extracted, 0 skipped (no delsac).
         → 66 delsac sessions

[  53.0s]  Filtering neurons ...
         → 15164 / 17668 neurons kept

[  55.5s]  Aligning to targetOn = 1.7 s (at 0.025 s precision) ...

[  56.2s]  Sliding window: window=0.1 s, step=0.025 s → 236 bins, 75% overlap ...
         → spikecounts shape: (154, 104, 236)  (trials × neurons × timebins)

[  65.5s]  Merging behavioral data ...

[  66.2s]  Removing NaN trials ...
[drop_empty_sessions] Dropped 3 session(s).

───────────────────────────────────────────────────────
Pipeline done in 67.0s  —  63 sessions ready for decoding.
───────────────────────────────────────────────────────

── Sessions & trials per subject ──────────────────
  Subject    Sessions   Trials
  Paula            43     4940
  Rex              20     2607
  TOTAL            63     7547

── Sessions & trials per nMapStim condition ────────
  nMapStim   Sessions   Trials
  6                45     6043
  8                18     1504

In [4]:
import pickle
from load_and_filter import load_pipeline

def load_data(filepath):
    with open(filepath, "rb") as f:
        data = pickle.load(f)
    return data

data_path = "/home/aarghavan/aslan/data/test.pkl"
data = load_data(data_path)

print(type(data))
if isinstance(data, dict):
    print(data.keys())
else:
    print(data)

<class 'dict'>
dict_keys(['trial', 'unit', 'session', 'spikecounts', 'trialsNeuron'])


In [5]:
len(data['spikecounts'][4][0][0])

58

In [9]:
"""
decoder.py
==========
Neural decoding pipeline for the distributed working memory project.

Input
-----
  A filtered data dict produced by load_and_filter.load_pipeline(), stored as a PKL.
  Expected keys per session index s:
    data['spikecounts'][s]   ndarray (n_trials, n_neurons, n_time)
    data['trial'][s]         DataFrame  —  must contain targetX/Y, responseX/Y,
                                           plus behavioral columns from behavior_all.csv
    data['unit'][s]          DataFrame  —  must contain 'area' column

  The spikecounts are assumed to be sliding-window bins (window=0.1 s, step=0.025 s),
  giving n_time = 237 bins at 0.025 s effective resolution.
  Call load_pipeline() with use_sliding_window=True (the default) to produce this.

Outputs  (all written to RESULTS_DIR)
--------------------------------------
  smoothed.pkl              cached smoothed spikecounts  (recomputed if absent)
  centered.pkl              cached z-scored spikecounts + filtered unit metadata
  predicted_data.pkl        raw LOO predictions  [area][session][var]['predictions']
  shuffled_data.pkl         null (label-permuted) predictions  [area][session][var]
  per_session_circcorr.pkl  {'real': ..., 'null': ...}  per-session circ-corr curves
  errors_angular.csv        per-trial neural decoding error summary (one row per trial)
  neurobeh_baseline.pkl     comprehensive package consumed by neurobehCorr.ipynb
  circ_correlation.svg      time-resolved circular correlation, all areas
  prediction_errors.svg     time-resolved |angular error|, all areas
"""

# ─── stdlib / third-party ──────────────────────────────────────────────────────
import os
import sys
import pickle
import itertools
import time as _time
from math import ceil
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.signal
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import LeaveOneOut, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from joblib import Parallel, delayed
from tqdm import tqdm
from astropy.stats import circcorrcoef


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG  ← edit paths and parameters here
# ══════════════════════════════════════════════════════════════════════════════

DATA_PATH   = '/home/aarghavan/aslan/data/test.pkl'
RESULTS_DIR = '/home/aarghavan/aslan/delsac-neural-decoding/results/'

# ── Time-axis parameters (must match load_and_filter.py settings) ─────────────
T_START    = -3.5    # start of recording epoch (seconds)
ORIG_BIN   = 0.025   # raw bin size before sliding window (seconds)
N_BINS_RAW = 240     # number of raw bins in the epoch
WINDOW_S   = 0.2     # integration window width (seconds)
STEP_S     = 0.1     # step between bins (seconds)  — window > step → 50 % overlap → dependent bins
# Derived: n_time = (N_BINS_RAW − W) // S + 1  =  (240 − 4) // 4 + 1  =  59

# ── Smoothing (causal exponential kernel applied after sliding window) ─────────
SMOOTH_WIDTH = 1.5   # kernel temporal extent (seconds)
SMOOTH_K     = 2.0   # shape parameter (higher → faster decay)

# ── Decoding ──────────────────────────────────────────────────────────────────
AREAS        = ['PFC', 'FEF', 'LIP', 'Parietal', 'IT', 'MT', 'V4']
# AREAS        = ['PFC']
MIN_NEURONS  = 10      # minimum neurons per area per session to decode
N_SHUFFLES   = 20      # permutations for shuffled-prediction null (only used if COMPUTE_NULL=True)
COMPUTE_NULL = False   # set True to compute label-permuted predictions (rarely needed)
N_NULL_PERMS = 50      # permutations for angle-permutation null in circcorr (always computed)
CV_FOLDS     = 5       # KFold splits used when USE_LOO=False
USE_LOO      = False    # True → LeaveOneOut (exact, slow); False → KFold(CV_FOLDS) (~80x faster)
MAX_SESSIONS = None    # set to integer (e.g. 3) for quick testing; None = all

# ── Cross-temporal decoding ────────────────────────────────────────────────────
COMPUTE_CROSS_TEMPORAL = False   # train on bin t, test on all bins t' → (n_time × n_time) matrix

# ── Decoding time window ───────────────────────────────────────────────────────
DECODE_START = 0   # only decode from this time onwards (seconds); None = full epoch
DECODE_END   = None  # clip end as well (seconds); None = keep all

# ── Variables to decode and matching angle pairs ───────────────────────────────
# Edit VARIABLES_TO_DECODE to control what gets decoded.
# Keep ANGLE_PAIRS consistent: only include pairs where both X and Y are decoded.
VARIABLES_TO_DECODE = [
    'targetX', 'targetY',
    # 'responseX', 'responseY',
    # 'prevtargetX', 'prevtargetY',
    # 'prevresponseX', 'prevresponseY',
]

ANGLE_PAIRS = {
    'targetAngle':     ('targetX',       'targetY'),
    # 'respAngle':       ('responseX',     'responseY'),
    # 'prevTargetAngle': ('prevtargetX',   'prevtargetY'),
    # 'prevRespAngle':   ('prevresponseX', 'prevresponseY'),
}

# ── Event times for plots (seconds, relative to aligned epoch) ─────────────────
EV_TARGET_ON  = 1.70   # targetOn (alignment reference)
EV_TARGET_OFF = 1.80
EV_RESPONSE   = 2.55   # approximate fixptOff / saccade initiation

# ── Delay period for per-trial CSV summary ────────────────────────────────────
DELAY_START = 1.80   # targetOff
DELAY_END   = 2.55   # response

# ── Behavioral columns attached to errors_angular.csv ─────────────────────────
BEH_COLS_CSV = ['targAng', 'respAng_dva', 'err', 'folded_err', 'abs_err',
                 'memoryDelay', 'ITI']

os.makedirs(RESULTS_DIR, exist_ok=True)
SMOOTHED_PATH = os.path.join(RESULTS_DIR, 'smoothed.pkl')
CENTERED_PATH = os.path.join(RESULTS_DIR, 'centered.pkl')


# ══════════════════════════════════════════════════════════════════════════════
# TIME AXIS
# ══════════════════════════════════════════════════════════════════════════════

def make_time_axis(t_start=T_START, orig_bin=ORIG_BIN,
                   n_bins=N_BINS_RAW, window_s=WINDOW_S, step_s=STEP_S):
    """Return the sliding-window centre times (one per integration window)."""
    W = round(window_s / orig_bin)    # 4  bins
    S = round(step_s   / orig_bin)    # 1  bin
    n = (n_bins - W) // S + 1         # 237 windows
    return np.array([t_start + (i * S + W / 2) * orig_bin for i in range(n)])


# Time axis is computed AFTER loading data so n_time matches the actual array shape.
# Placeholder here; redefined below once data is loaded.
time      = None
n_time    = None
delay_idx = None


# ══════════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

print(f"\nLoading data from:\n  {DATA_PATH}")
with open(DATA_PATH, 'rb') as f:
    data = pickle.load(f)

num_sessions = len(data['spikecounts'])
print(f"  {num_sessions} sessions loaded.")
if num_sessions == 0:
    raise ValueError("No sessions found in data!")

print("\nVerifying spikecounts ↔ unit alignment:")
for s in range(num_sessions):
    sc  = np.asarray(data['spikecounts'][s])
    n_sc, n_u = sc.shape[1], len(data['unit'][s])
    if n_sc != n_u:
        raise ValueError(
            f"  Session {s}: spikecounts has {n_sc} neurons "
            f"but unit DataFrame has {n_u}."
        )
print("  OK\n")

# ── Build time axis from actual data shape (avoids N_BINS_RAW mismatch) ───────
_n_time_actual = np.asarray(data['spikecounts'][0]).shape[-1]
W = round(WINDOW_S / ORIG_BIN)
S = round(STEP_S   / ORIG_BIN)
time      = np.array([T_START + (i * S + W / 2) * ORIG_BIN
                       for i in range(_n_time_actual)])
n_time    = _n_time_actual
delay_idx = (time >= DELAY_START) & (time <= DELAY_END)
print(f"Time axis: {n_time} bins  [{time[0]:.3f} s → {time[-1]:.3f} s]  step = {STEP_S} s\n")



Loading data from:
  /home/aarghavan/aslan/data/test.pkl
  63 sessions loaded.

Verifying spikecounts ↔ unit alignment:
  OK

Time axis: 58 bins  [-3.400 s → 2.300 s]  step = 0.1 s



In [10]:
time[-1]

2.3000000000000007